# ⚙️ Pipeline: Optimización de Hiperparámetros

**Propósito:** Encontrar la mejor combinación de hiperparámetros (neuronas, learning rate, dropout)
usando Keras Tuner (si está instalado) o Grid Search manual como fallback.

---
## 🎯 ¿Qué son los hiperparámetros?

Son valores que **no** aprende el modelo durante el entrenamiento:

| Hiperparámetro | Qué controla | Valores típicos |
|----------------|-------------|-----------------|
| `units` | Capacidad de la capa | 32, 64, 128, 256 |
| `learning_rate` | Velocidad de convergencia | 1e-2, 1e-3, 1e-4 |
| `dropout` | Regularización (evita overfitting) | 0.2, 0.3, 0.5 |
| `batch_size` | Estabilidad del gradiente | 16, 32, 64 |
| `num_capas` | Profundidad de la red | 1, 2, 3 |

In [ ]:
# MACHOTE — Setup universal
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from modulo4_libreria import *

INFO = setup_completo()

In [ ]:
# MACHOTE — Dataset sintético de prueba

from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=15,
    n_redundant=5, n_classes=2, random_state=42
)

data = train_val_test_split(X, y, test_size=0.2, val_size=0.1)

X_train, X_val, X_test = data['X_train'], data['X_val'], data['X_test']
y_train, y_val, y_test = data['y_train'], data['y_val'], data['y_test']

print(f"Features: {X.shape[1]}, Clases: {len(np.unique(y))}")

In [ ]:
# MACHOTE — Modelo base (valores por defecto) como baseline

modelo_default = crear_mlp(
    input_shape=(X.shape[1],),
    num_clases=2,
    capas_ocultas=[128, 64],
    dropout=0.3
)

hist_default = compilar_y_entrenar(
    modelo_default,
    X_train, y_train,
    X_val, y_val,
    num_clases=2,
    lr=0.001,
    epochs=50,
    early_stop_paciencia=5
)

print(f"\n📊 Baseline — Accuracy en validación: {max(hist_default.history['val_accuracy']):.4f}")

---
## 🧪 Keras Tuner (si está instalado)

Keras Tuner realiza búsqueda automática de hiperparámetros con Random Search,
Hyperband o Bayesian Optimization.

In [ ]:
# MACHOTE — Keras Tuner: definición del modelo de búsqueda

KERAS_TUNER_DISPONIBLE = False
try:
    import keras_tuner as kt
    KERAS_TUNER_DISPONIBLE = True
    print("✅ Keras Tuner detectado")
except ImportError:
    print("⚠️ Keras Tuner no instalado — se usará Grid Search manual")

In [ ]:
# MACHOTE — Búsqueda con Keras Tuner (solo si está disponible)

if KERAS_TUNER_DISPONIBLE:
    
    def build_model(hp):
        modelo = keras.Sequential(name="MLP_Tuneado")
        modelo.add(layers.Input(shape=(X.shape[1],)))
        
        # Hiperparámetros a muestrear
        units_1 = hp.Int('units_1', 32, 256, step=32)
        dropout = hp.Choice('dropout', [0.2, 0.3, 0.5])
        lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
        
        modelo.add(layers.Dense(units_1, activation='relu'))
        modelo.add(layers.BatchNormalization())
        modelo.add(layers.Dropout(dropout))
        modelo.add(layers.Dense(64, activation='relu'))
        modelo.add(layers.BatchNormalization())
        modelo.add(layers.Dropout(dropout))
        modelo.add(layers.Dense(1, activation='sigmoid'))
        
        modelo.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        return modelo
    
    tuner = kt.RandomSearch(
        build_model,
        objective='val_accuracy',
        max_trials=10,
        executions_per_trial=1,
        directory='outputs',
        project_name='hiperparametros'
    )
    
    print("🔍 Buscando mejores hiperparámetros (10 trials)...")
    tuner.search(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
        verbose=1
    )
    
    mejores_hp = tuner.get_best_hyperparameters(1)[0]
    print("\n🏆 Mejores hiperparámetros encontrados:")
    for param in ['units_1', 'learning_rate', 'dropout']:
        print(f"   {param}: {mejores_hp.get(param)}")
    
    modelo_tuneado = tuner.get_best_models(1)[0]
    
else:
    print("\n--- Usando Grid Search Manual ---")

In [ ]:
# MACHOTE — Grid Search Manual (fallback / siempre funciona)

if not KERAS_TUNER_DISPONIBLE:
    
    unidades_opciones = [64, 128]
    lr_opciones = [1e-2, 1e-3]
    dropout_opciones = [0.2, 0.3, 0.5]
    
    resultados = []
    mejor_acc = 0
    mejores_params = None
    
    print(f"🔍 Grid Search: {len(unidades_opciones) * len(lr_opciones) * len(dropout_opciones)} combinaciones\n")
    
    for u in unidades_opciones:
        for lr in lr_opciones:
            for d in dropout_opciones:
                print(f"   Probando: units={u}, lr={lr}, dropout={d}... ", end='')
                modelo = crear_mlp(
                    input_shape=(X.shape[1],), num_clases=2,
                    capas_ocultas=[u, 64], dropout=d
                )
                hist = compilar_y_entrenar(
                    modelo, X_train, y_train, X_val, y_val,
                    num_clases=2, lr=lr, epochs=30, early_stop_paciencia=3, verbose=0
                )
                val_acc = max(hist.history['val_accuracy'])
                resultados.append({'units': u, 'lr': lr, 'dropout': d, 'val_acc': val_acc})
                print(f"acc={val_acc:.4f}")
                
                if val_acc > mejor_acc:
                    mejor_acc = val_acc
                    mejores_params = {'units': u, 'lr': lr, 'dropout': d}
                    modelo_tuneado = modelo
    
    print("\n" + "="*50)
    print("🏆 Mejores hiperparámetros (Grid Search):")
    for k, v in mejores_params.items():
        print(f"   {k}: {v}")
    print(f"   val_acc: {mejor_acc:.4f}")
    
    # Mostrar ranking
    df_resultados = pd.DataFrame(resultados).sort_values('val_acc', ascending=False)
    print("\n📊 Ranking completo:")
    print(df_resultados.to_string(index=False))

In [ ]:
# MACHOTE — Comparar modelo óptimo vs default

print("=" * 50)
print("📊 COMPARACIÓN: Default vs Tuneado")
print("=" * 50)

acc_default = max(hist_default.history['val_accuracy'])

# Evaluar modelo tuneado
y_pred_prob = modelo_tuneado.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
acc_tuneado = np.mean(y_pred == y_test)

print(f"\n   Baseline (default): {acc_default:.4f}")
print(f"   Modelo optimizado:   {acc_tuneado:.4f}")
print(f"   Mejora:              {((acc_tuneado - acc_default) / acc_default) * 100:+.2f}%")

# Gráfica comparativa
fig, axes = plt.subplots(1, 2, figsize=(14,5))

for ax, hist, titulo in zip(
    axes,
    [hist_default, hist],
    ['Default [128,64], lr=0.001, dropout=0.3', 'Modelo Optimizado']
):
    ax.plot(hist.history['loss'], label='Train Loss', color='steelblue')
    ax.plot(hist.history['val_loss'], label='Val Loss', color='tomato', linestyle='--')
    ax.set_title(titulo)
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Comparación: Curvas de Pérdida', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📖 Interpretación de resultados

### ¿Qué nos dicen los mejores hiperparámetros?

- **units más grandes** → más capacidad, riesgo de overfitting
- **learning_rate más bajo** → convergencia más lenta pero estable
- **dropout más alto** → más regularización, evita overfitting

### Reglas generales

| Valor | Learning Rate | Dropout | Units |
|-------|--------------|---------|-------|
| Alto | Oscila/diverge | Mucha regularización | Sobreajuste |
| Bajo | Converge lento | Poca regularización | Subajuste |
| Recomendado | 1e-3 (Adam) | 0.2–0.5 | 64–256 |

### 📌 Nota importante

La optimización de hiperparámetros debe hacerse **solo sobre el conjunto de validación**.
El test set se usa **una sola vez** al final para reportar el resultado real.

---
*Pipeline creado para el Diplomado en Redes Neuronales — Módulo 4* 🧠

In [ ]:
# TODO: Experimenta con estos hiperparámetros adicionales
# 1. Agrega batch_size como hiperparámetro (16, 32, 64)
# 2. Agrega num_capas como hiperparámetro (1, 2, 3)
# 3. Prueba con optimizadores diferentes: 'adam', 'sgd', 'rmsprop'
# 4. ¿El mejor combo encontrado funciona igual en otro dataset?
# Pregunta: ¿Cuánto mejora realmente la optimización vs el modelo default?